# [서연] TOPIS 월별 교통량 수집·전처리 실행 기록

실제 기능은 crawling/traffic_volume.py에 두고, 이 노트북에서는 코드 버전·과거 성공/실패/중단 기록·오류·재시도·화면 캡처·결과 표를 확인한다.

기록 조회 셀은 Selenium을 실행하지 않는다. 1개월 시험은 EXECUTE_ONE_MONTH_TEST, 전체 수집은 EXECUTE_COLLECTION 값을 직접 True로 바꾼 경우에만 시작한다.

## [서연] 1. 커널·패키지·프로젝트 경로 확인

In [4]:
# [서연] 환경 확인 — Selenium·네트워크·파일 저장 없음
import importlib.util
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'README.md').is_file() and (candidate / 'crawling' / 'traffic_volume.py').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
packages = ['selenium', 'pandas', 'openpyxl', 'pyarrow']
package_status = {name: importlib.util.find_spec(name) is not None for name in packages}
print('Python:', sys.version)
print('실행 파일:', sys.executable)
print('프로젝트:', ROOT)
print('패키지:', package_status)

Python: 3.11.18 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 18:13:29) [MSC v.1942 64 bit (AMD64)]
실행 파일: C:\Users\USER\.conda\envs\youth-isolation\python.exe
프로젝트: C:\Users\USER\seoul-weather-traffic
패키지: {'selenium': True, 'pandas': True, 'openpyxl': True, 'pyarrow': True}


## [서연] 2. 핵심 모듈과 코드 버전 확인

In [5]:
# [서연] 핵심 함수 코드와 Git 상태 표시 — 수집 실행 없음
import inspect
from crawling import traffic_volume as TV

provenance_now = TV.git_provenance()
display(provenance_now)
for function in [TV.required_months, TV.download_required_months, TV.preprocess_downloads, TV.run_pipeline]:
    print('\n' + '=' * 90)
    print(function.__name__)
    print('=' * 90)
    print(inspect.getsource(function))

{'git_available': True,
 'git_commit': '85b44cbba524d62889bdb6a1d7a7ecc609f0432d',
 'git_branch': 'c-traffic',
 'git_dirty': True,
 'git_changed_files': ['rawling/traffic_volume.py',
  'notebooks/10_B_comparison.ipynb',
  'notebooks/yang_seoyeon/yang_seoyeon_05_traffic_volume.ipynb',
  'tests/ysy/test_traffic_volume_finalize.py',
  'data/sample/yang_seoyeon_topis_weather_commute_sample_20261122_v01.csv',
  'data/sample/yang_seoyeon_topis_weather_commute_sample_20261122_v02.csv',
  'data/sample/yang_seoyeon_topis_weather_link_sample_20261122_v01.csv',
  'outputs/tables/yang_seoyeon_roads_candidate_review_20261122_v01.csv',
  'outputs/tables/yang_seoyeon_stable_links_20261122_v01.csv',
  'outputs/tables/yang_seoyeon_topis_partial_missing_review_20261122_v01.csv',
  'outputs/tables/yang_seoyeon_topis_weather_merge_summary_20261122_v01.csv',
  'outputs/tables/yang_seoyeon_topis_weather_merge_summary_20261122_v02.csv'],
 'code_sha256': {'crawling\\traffic_volume.py': '71ae34f4f673b1d4c6ea95


required_months
def required_months(days: pd.DataFrame | None = None) -> list[str]:
    """90일 목록에서 중복 없는 YYYY-MM 26개를 정렬해 반환한다."""
    target = load_target_days() if days is None else days
    months = sorted(target["date"].dt.strftime("%Y-%m").unique().tolist())
    if len(months) != 26:
        raise ValueError(f"필요 연월은 현재 기준 26개여야 합니다: 실제 {len(months)}개")
    return months


download_required_months
def download_required_months(
    *,
    root: Path | None = None,
    months: Iterable[str] | None = None,
    headless: bool = False,
    retries: int = 3,
    wait_seconds: int = 60,
    run_dir: Path | None = None,
    run_id: str | None = None,
    progress: Callable[[str], None] = print,
) -> dict[str, object]:
    """TOPIS 화면을 조작해 필요한 월별 XLSX를 다운로드한다.

    이미 ``topis_traffic_volume_YYYY-MM.xlsx``가 있으면 크기가 0이 아닌지
    확인하고 건너뛴다. 실패한 월은 제한 횟수만 재시도하고 다음 월로 진행한다.
    """
    base = (root or data_root()).resolve()
    raw_dir = base / RAW_SUBDIR
    record_dir = run_dir or (base / LOG

## [서연] 3. 정희의 90일과 필요한 26개월 확인

In [3]:
# [서연] 대상 기간 확인 — 수집 실행·파일 저장 없음
target_days = TV.load_target_days()
target_months = TV.required_months(target_days)
display(target_days.groupby(['split', 'role']).size().rename('days').reset_index())
print('날짜 범위:', target_days.date.min().date(), '~', target_days.date.max().date())
print('필요 연월:', len(target_months), '개')
print(target_months)

,split,role,days
0,eval,비교일,15
1,eval,악천후일,15
2,train,비교일,30
3,train,악천후일,30


날짜 범위: 2022-12-21 ~ 2026-09-08
필요 연월: 26 개
['2022-12', '2023-01', '2023-06', '2023-07', '2023-08', '2023-11', '2023-12', '2024-01', '2024-02', '2024-07', '2024-11', '2024-12', '2025-02', '2025-03', '2025-07', '2025-08', '2025-09', '2025-11', '2025-12', '2026-01', '2026-02', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09']


## [서연] 4. 과거 실행 이력 전체 조회
성공 기록만 고르지 않고 성공·실패·중단 폴더를 모두 읽는다. 이 셀을 다시 실행해도 기존 기록을 변경하지 않는다.

In [4]:
# [서연] 과거 실행 요약 — Selenium·네트워크·파일 저장 없음
history = TV.list_run_history()
display(history if not history.empty else '아직 실행 기록이 없습니다.')

'아직 실행 기록이 없습니다.'

## [서연] 5. 오류·재시도 로그 전체 조회

In [5]:
# [서연] 모든 실행의 JSONL 로그 표시 — 수집 실행 없음
import json
import pandas as pd

run_roots = [TV.RUN_DIR, TV.data_root() / TV.RUN_SUBDIR, TV.data_root() / TV.LEGACY_RUN_SUBDIR]
log_rows = []
for run_root in run_roots:
    if not run_root.is_dir():
        continue
    log_paths = {*run_root.glob('**/collection_log_*.jsonl'), *run_root.glob('**/yang_seoyeon_collection_log_*.jsonl')}
    for path in sorted(log_paths):
        for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                row = {'status': 'log_parse_error', 'error': str(exc)}
            row['run_folder'] = path.parent.name
            row['line_number'] = line_number
            log_rows.append(row)
log_history = pd.DataFrame(log_rows)
display(log_history if not log_history.empty else '아직 수집 로그가 없습니다.')
if not log_history.empty:
    display(log_history.groupby(['run_folder', 'status']).size().rename('count').reset_index())

'아직 수집 로그가 없습니다.'

## [서연] 6. 과거 실행 기록 안내

In [6]:
# [서연] 신규 스크린샷 저장은 사용하지 않으며 기존 기록은 삭제하지 않음
print('작업 과정은 실행 로그·재시도·오류·검증표로 확인합니다.')

화면 캡처: 0 개


## [서연] 7. 1개월 테스트 명시적 실행
전체 수집 전에 필요한 26개월 중 한 달만 시험한다. EXECUTE_ONE_MONTH_TEST를 직접 True로 바꾼 경우에만 Selenium이 실행된다. 성공·실패·중단·재시도 기록은 전체 실행과 같은 방식으로 별도 보존된다.

In [7]:
# [서연] 1개월 Selenium 테스트 — 기본값 False이므로 자동 실행되지 않음
EXECUTE_ONE_MONTH_TEST = False
TEST_MONTH = '2024-11'

if not EXECUTE_ONE_MONTH_TEST:
    print('1개월 테스트를 실행하지 않았습니다. 실행할 때만 True로 변경하세요.')
else:
    one_month_test_result = TV.run_test_month(
        TEST_MONTH,
        headless=False,
        retries=3,
        wait_seconds=60,
        progress=print,
    )
    display(one_month_test_result)

[1개월 시험 시작] 2024-11: D:\seoul-weather-traffic-data\traffic_volume\runs\yang_seoyeon\yang_seoyeon_traffic_volume_test_2024-11_20261123T161744_v01_(중단)
[완료] 2024-11: yang_seoyeon_topis_traffic_volume_2024-11.xlsx (1,353,063 bytes)
[1개월 시험 실패] D:\seoul-weather-traffic-data\traffic_volume\runs\yang_seoyeon\yang_seoyeon_traffic_volume_test_2024-11_20261123T161744_v01_(실패)


RuntimeError: 1개월 시험 실패 기록: D:\seoul-weather-traffic-data\traffic_volume\runs\yang_seoyeon\yang_seoyeon_traffic_volume_test_2024-11_20261123T161744_v01_(실패); 원인: {'type': 'ImportError', 'message': '`Import openpyxl` failed.  Use pip, conda, or your preferred package management tool to install the openpyxl package.'}

## [서연] 8. 전체 26개월 수집·전처리 명시적 실행
이 셀만 Selenium, 네트워크, XLSX 다운로드, Parquet 저장을 발생시킨다. 실행하려면 EXECUTE_COLLECTION을 직접 True로 바꾼다. 전용 Chrome이 닫힐 때까지 창을 조작하지 않는다.

In [ ]:
# [서연] 실제 수집 실행 셀 — 기본값 False이므로 자동 실행되지 않음
EXECUTE_COLLECTION = False

if not EXECUTE_COLLECTION:
    print('수집하지 않았습니다. 직접 실행할 때만 EXECUTE_COLLECTION = True로 변경하세요.')
else:
    run_result = TV.run_pipeline(
        headless=False,
        retries=3,
        wait_seconds=60,
        progress=print,
    )
    display(run_result)

## [서연] 9. 실행 후 이력·검증·결과 표 새로 읽기
실제 수집이 끝난 뒤 이 셀을 실행하고 노트북을 저장한다. 이전 실패 기록까지 포함해 교수님이 확인할 수 있다.

In [ ]:
# [서연] 실행 후 결과 요약 — Selenium·네트워크·파일 저장 없음
history_after = TV.list_run_history()
display(history_after if not history_after.empty else '실행 기록이 없습니다.')

if not history_after.empty:
    for run_dir_text in history_after.run_dir:
        run_dir = Path(run_dir_text)
        for validation_path in sorted(run_dir.glob('yang_seoyeon_*validation*.csv')):
            print(run_dir.name, '/', validation_path.name)
            display(pd.read_csv(validation_path, encoding='utf-8-sig'))
        sample_paths = {*run_dir.glob('test_sample_*.csv'), *run_dir.glob('yang_seoyeon_test_sample_*.csv')}
        for sample_path in sorted(sample_paths):
            print(run_dir.name, '/', sample_path.name)
            display(pd.read_csv(sample_path, encoding='utf-8-sig').head(20))
        summary_paths = {*run_dir.glob('summary_*.json'), *run_dir.glob('yang_seoyeon_summary_*.json')}
        for summary_path in sorted(summary_paths):
            print(run_dir.name, '/', summary_path.name)
            display(json.loads(summary_path.read_text(encoding='utf-8')))

## [서연] 10. 수집 데이터와 최종 결과 소량 확인

In [ ]:
# [서연] 원본 목록과 최종 Parquet 집계·미리보기 — 읽기 전용
raw_files = sorted((TV.data_root() / TV.RAW_SUBDIR).glob('20??-??.xlsx'))
raw_table = pd.DataFrame([{'file': p.name, 'bytes': p.stat().st_size, 'modified_at': pd.Timestamp(p.stat().st_mtime, unit='s')} for p in raw_files])
display(raw_table)

processed_files = sorted(TV.processed_root().glob('yang_seoyeon_traffic_volume_*_(성공).parquet'))
for path in processed_files:
    frame = pd.read_parquet(path)
    print(path.name, '행:', len(frame), '날짜:', frame.date.nunique(), '지점:', frame.spot_id.nunique())
    display(frame.head(10))
    display(pd.DataFrame([{
        'traffic_volume_missing': int(frame.traffic_volume.isna().sum()),
        'coordinate_missing': int(frame[['latitude', 'longitude']].isna().any(axis=1).sum()),
        'duplicate_keys': int(frame.duplicated(['date', 'spot_id', 'io_type', 'hour']).sum()),
    }]))

## [서연] 11. 교통량 결측 분포 확인
최신 성공 train·eval Parquet를 읽어 결측이 날짜·지점·시간대에 집중되는지 확인한다. 이 셀은 Selenium·네트워크·파일 저장을 실행하지 않는다.

In [6]:
# [서연] 최신 성공 결과의 결측 분포 분석 — 읽기 전용
import pandas as pd
processed_dir = TV.processed_root()
train_files = sorted(processed_dir.glob('traffic_volume_train_*_(성공).parquet'))
eval_files = sorted(processed_dir.glob('traffic_volume_eval_*_(성공).parquet'))
if not train_files or not eval_files:
    raise FileNotFoundError('성공한 train/eval Parquet를 찾지 못했습니다.')
latest = pd.concat([pd.read_parquet(train_files[-1]), pd.read_parquet(eval_files[-1])], ignore_index=True)
missing_report = TV.summarize_traffic_volume_missing(latest)
print('전체 요약')
display(missing_report['summary'])
print('학습·평가 구분')
display(missing_report['by_split'])
print('결측률 상위 연월')
display(missing_report['by_source_month'].head(15))
print('결측률 상위 날짜')
display(missing_report['by_date'].head(20))
print('시간대별')
display(missing_report['by_hour'])
print('결측률 상위 지점·방향')
display(missing_report['by_spot'].head(30))
print('결측 행 예시')
display(missing_report['missing_sample'].head(30))

전체 요약


,rows,missing_rows,missing_pct,dates,spots
0,125100,22774,18.205,90,139


학습·평가 구분


,split,rows,missing_rows,missing_pct
0,train,83400,17012,20.398
1,eval,41700,5762,13.818


결측률 상위 연월


,source_month,rows,missing_rows,missing_pct
0,2024-11,2780,1061,38.165
1,2022-12,1390,342,24.604
2,2023-01,1390,337,24.245
3,2023-11,1390,321,23.094
4,2025-03,1390,318,22.878
5,2025-08,5560,1242,22.338
6,2024-02,15290,3245,21.223
7,2024-01,6950,1469,21.137
8,2023-12,4170,875,20.983
9,2024-07,8340,1749,20.971


... (4 outputs left out)

결측률 상위 지점·방향


,spot_id,spot_name,io_type,rows,missing_rows,missing_pct
0,D-26,시흥대로(구로디지털단지역),유출,450,372,82.667
1,D-46,서부간선도로,유출,450,289,64.222
2,D-46,서부간선도로,유입,450,284,63.111
3,B-02,동일로(의정부IC),유입,450,273,60.667
4,B-24,금천교(광명시계),유입,450,244,54.222
5,B-13,밤고개로(세곡동사거리),유입,450,240,53.333
6,B-24,금천교(광명시계),유출,450,240,53.333
7,B-13,밤고개로(세곡동사거리),유출,450,235,52.222
8,B-25,금하로(광명시계),유입,450,235,52.222
9,B-16,양재대로(양재IC),유입,450,227,50.444


결측 행 예시


,date,split,spot_id,spot_name,io_type,hour,source_month,source_file
0,2022-12-21,train,A-02,사직로(사직터널),유입,8,2022-12,yang_seoyeon_topis_traffic_volume_2022-12.xlsx
1,2022-12-21,train,A-02,사직로(사직터널),유입,9,2022-12,yang_seoyeon_topis_traffic_volume_2022-12.xlsx
2,2022-12-21,train,A-02,사직로(사직터널),유입,18,2022-12,yang_seoyeon_topis_traffic_volume_2022-12.xlsx
3,2022-12-21,train,A-02,사직로(사직터널),유입,19,2022-12,yang_seoyeon_topis_traffic_volume_2022-12.xlsx
4,2022-12-21,train,A-02,사직로(사직터널),유입,20,2022-12,yang_seoyeon_topis_traffic_volume_2022-12.xlsx
5,2022-12-21,train,A-02,사직로(사직터널),유출,8,2022-12,yang_seoyeon_topis_traffic_volume_2022-12.xlsx
6,2022-12-21,train,A-02,사직로(사직터널),유출,9,2022-12,yang_seoyeon_topis_traffic_volume_2022-12.xlsx
7,2022-12-21,train,A-02,사직로(사직터널),유출,18,2022-12,yang_seoyeon_topis_traffic_volume_2022-12.xlsx
8,2022-12-21,train,A-02,사직로(사직터널),유출,19,2022-12,yang_seoyeon_topis_traffic_volume_2022-12.xlsx
9,2022-12-21,train,A-02,사직로(사직터널),유출,20,2022-12,yang_seoyeon_topis_traffic_volume_2022-12.xlsx


## [서연] 12. 제출 전 저장
조회 셀과 9~11번 결과 셀을 실행한 뒤 Jupyter의 Save 기능으로 노트북을 저장한다. 실행 번호·오류·검증표·결측 분석 출력이 노트북에 남는다. 실제 수집 셀은 다시 실행하지 않아도 과거 실행 폴더를 통해 기록을 계속 확인할 수 있다.